# 🕉️ SWAYAMBHU v2 — High-Speed GPU Ingestion on Google Colab

This notebook runs the **SWAYAMBHU v2 Discourse Ingestion Pipeline** on Google Colab with **GPU Acceleration**.

### ⚡ Why run on Google Colab GPU?
- **20x - 50x Faster**: Uses NVIDIA CUDA (Tesla T4) for `faster-whisper` (large-v3) speech-to-text and `sentence-transformers` batch embeddings.
- **Gigabit Bandwidth**: YouTube video metadata & audio downloads happen at 100+ MB/s on Google's backbone.
- **Direct Cloud Sync**: Vector embeddings and sliding-window timestamp chunks are saved directly into your cloud **Supabase pgvector** database.
- **Automatic Resume**: Already-indexed discourses are automatically skipped; only new discourses are processed.

### Step 1: Verify GPU & Hardware
Make sure Colab is set to GPU: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Check NVIDIA GPU availability
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected! Go to Runtime > Change runtime type > Select T4 GPU.")

### Step 2: Install Ingestion Dependencies
Installs `yt-dlp`, `youtube-transcript-api`, `faster-whisper`, `sentence-transformers`, `supabase`, `pydantic-settings`, and `groq`.
*(FFmpeg and PyTorch CUDA are already pre-installed on Google Colab)*.

In [ ]:
# Clean install without conflicting with Colab's pre-installed packages
!pip install -q yt-dlp youtube-transcript-api faster-whisper sentence-transformers supabase pydantic-settings groq
print('✅ Dependencies installed successfully!')

### Step 3: Unpack SWAYAMBHU v2 Pipeline Code
The pipeline source files are bundled below and unpacked automatically.

In [ ]:
import base64
import os
import zipfile

ZIP_B64 = """..."""

with open('swayambhu_package.zip', 'wb') as f:
    f.write(base64.b64decode(ZIP_B64))

with zipfile.ZipFile('swayambhu_package.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print('✅ Unpacked SWAYAMBHU v2 pipeline files:')
!ls -lh *.py

### Step 4: Configure Supabase & API Credentials
These credentials connect directly to your Supabase pgvector database.

In [ ]:
import os

# Supabase & Cloud API Settings (Replace with your actual keys)
os.environ['APP_ENV'] = 'production'
os.environ['VECTOR_STORE_BACKEND'] = 'supabase'
os.environ['SUPABASE_URL'] = os.environ.get('SUPABASE_URL', 'https://your-project.supabase.co')
os.environ['SUPABASE_KEY'] = os.environ.get('SUPABASE_KEY', 'your-supabase-anon-key')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = os.environ.get('SUPABASE_SERVICE_ROLE_KEY', 'your-supabase-service-role-key')
os.environ['GROQ_API_KEY'] = os.environ.get('GROQ_API_KEY', 'your-groq-api-key')

# Embedding & Device Configuration
os.environ['EMBEDDING_MODEL_NAME'] = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
os.environ['EMBEDDING_DIM'] = '384'
os.environ['LOCAL_WHISPER_GPU_MODEL'] = 'large-v3'

print('✅ Environment credentials configured successfully!')

### Step 5: Check Current Database Stats Before Ingesting

In [ ]:
from supabase import create_client

client = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_ROLE_KEY'])
count_res = client.table('transcript_chunks').select('id', count='exact').limit(1).execute()
total_chunks = count_res.count if hasattr(count_res, 'count') else 'Unknown'

print(f'📊 Current Total Indexed Chunks in Supabase: {total_chunks}')

### Step 6: Launch GPU-Accelerated Ingestion

Choose the command that fits your goal:

- **Option A (The Other 3 Channels — Sadhan Path, Vrindavan Ras, Shri Hit Radha Kripa)**:
  `!python ingest.py --channel sadhan_path,vrindavan_ras,shri_hit_radha_kripa --max-videos 0`

- **Option B (All 4 Channels in Interleaved Round-Robin)**:
  `!python ingest.py --channel all --max-videos 0 --batch-size 10`

- **Option C (Sadhan Path Only)**:
  `!python ingest.py --channel sadhan_path --max-videos 50`

- **Option D (Vrindavan Ras Mahima Only)**:
  `!python ingest.py --channel vrindavan_ras --max-videos 50`

*(Videos already in Supabase are automatically skipped with `⏩ [SKIP]`)*

In [ ]:
# Run ingestion for the other 3 channels (or change --channel as needed):
!python ingest.py --channel sadhan_path,vrindavan_ras,shri_hit_radha_kripa --max-videos 0

### Step 7: Verify Indexed Chunks After Ingestion

In [ ]:
count_res = client.table('transcript_chunks').select('id', count='exact').limit(1).execute()
print(f'🎉 Updated Total Indexed Chunks in Supabase: {count_res.count}')